In [1]:
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0)


In [2]:
# Task 1 - Prompt
prompt_template = ChatPromptTemplate.from_messages([
    ('system', "You are a movie summarizer."),
    ('human', "Please summarise the movie in brief : {input}")
])

In [3]:
#Task 2 - Parser

str_parser = StrOutputParser()

In [8]:
#Task 3 - Custom Runnable
from langchain_core.runnables import RunnableLambda, RunnableParallel

def dictionary_maker(text : str) -> dict:
    return {"text": text}

dictionary_maker_runnable = RunnableLambda(dictionary_maker)

Parallel chain 1

In [6]:
linkedin_prompt = ChatPromptTemplate.from_messages([
   ("system", "You are a linkedin post generator."),
   ("human", "Generate a linkedin post for the following content : {text}")
])

chain_linkedin = linkedin_prompt | llm | str_parser

Parallel Chain 2

In [7]:
def insta_chain(text: dict):
    insta_prompt = ChatPromptTemplate.from_messages([
      ("system", "You are a Instagram post generator."),
      ("human", "Generate a Instagram post for the following content : {text}")
    ])

    chain_insta = insta_prompt | llm | str_parser

    result = chain_insta.invoke(text)

    return result

insta_chain_runnable = RunnableLambda(insta_chain)

Final

In [9]:
final_chain = (
               prompt_template |
               llm |
               str_parser |
               dictionary_maker_runnable |
               RunnableParallel(branches = {"linkedin": chain_linkedin, "insta": insta_chain_runnable})
)

In [10]:
final_chain.invoke("Oppenheimer")

{'branches': {'linkedin': 'Here are a few options for a LinkedIn post, choose the one that best fits your tone!\n\n---\n\n**Option 1 (Focus on Leadership & Ethics):**\n\nChristopher Nolan\'s #Oppenheimer isn\'t just a film; it\'s a profound case study in leadership, innovation, and the immense weight of moral responsibility.\n\nThe movie brilliantly chronicles J. Robert Oppenheimer\'s journey leading the top-secret Manhattan Project, a testament to human ingenuity pushed to its absolute limits. But beyond the scientific triumph lies a harrowing exploration of the ethical dilemmas faced when creating world-altering power.\n\nOppenheimer\'s post-war struggle with the implications of his creation, his advocacy for international control, and his subsequent political downfall due to personal vendettas and paranoia, offer chilling lessons. It\'s a stark reminder of how ambition, genius, and political maneuvering can intertwine, leaving even the most influential figures vulnerable.\n\nWhat ar

In [12]:
def beautify(final_response: dict) -> dict:
    linkedin_response = final_response['branches']['linkedin']
    insta_response = final_response['branches']['insta']
    return {"Linkedin": linkedin_response, "Instagram": insta_response}
beautify_runnable = RunnableLambda(beautify)

beautified_chain = final_chain | beautify_runnable

beautified_chain.invoke("Oppenheimer")
    

{'Linkedin': 'Here are a few options for a LinkedIn post about "Oppenheimer," playing on different angles:\n\n---\n\n**Option 1 (Focus on Leadership & Ethics):**\n\nBeyond the cinematic spectacle, "Oppenheimer" offers a profound case study in leadership, innovation, and ethical responsibility.\n\nThe film meticulously chronicles J. Robert Oppenheimer\'s journey leading the top-secret Manhattan Project, from the immense scientific ambition to the devastating reality of the Trinity test and its aftermath. It\'s a powerful exploration of the moral reckoning faced by the "father of the atomic bomb" and his subsequent fall from grace due to political machinations.\n\nThis story isn\'t just history; it\'s a stark reminder of the complex interplay between scientific advancement, ethical dilemmas, and political power.\n\nWhat leadership lessons or ethical considerations did you take away from Oppenheimer\'s story? Share your thoughts below. 👇\n\n#Oppenheimer #Leadership #Ethics #Innovation #Hi